# STLab02 – EduCommerce test execution
Chạy lần lượt từng cell từ trên xuống (hoặc Runtime → Run all). Cell nào có `[E0x]` là cell để chụp ảnh minh chứng.

In [ ]:
%%writefile ecommerce_app.py
"""EduCommerce: in-memory application for practicing checkout and automated testing."""
from dataclasses import dataclass, field
from decimal import Decimal, ROUND_HALF_UP
from typing import Optional

D = lambda x: Decimal(str(x))


def money(x):
    return D(x).quantize(D('1'), rounding=ROUND_HALF_UP)


@dataclass
class Product:
    sku: str
    title: str
    price: Decimal
    stock: int
    active: bool = True


@dataclass
class Customer:
    customer_id: str
    authenticated: bool = True
    used_codes: set = field(default_factory=set)


@dataclass
class CartLine:
    sku: str
    quantity: int


@dataclass
class Voucher:
    code: str
    percent: int
    min_subtotal: Decimal
    cap: Decimal
    active: bool = True


@dataclass
class Order:
    order_id: str
    customer_id: str
    total: Decimal
    status: str
    lines: list
    discount: Decimal
    shipping: Decimal
    payment_method: str


class Catalog:
    def __init__(self):
        self.products = {
            'A': Product('A', 'Wireless Mouse', D(200000), 20),
            'B': Product('B', 'Mechanical Keyboard', D(300000), 15),
            'C': Product('C', 'USB-C Hub', D(150000), 12)}

    def get(self, sku):
        p = self.products.get(sku)
        if not p or not p.active:
            raise ValueError('PRODUCT_UNAVAILABLE')
        return p


class Cart:
    def __init__(self, catalog):
        self.catalog = catalog
        self.lines = []

    def add(self, sku, quantity):
        self.catalog.get(sku)
        if not isinstance(quantity, int) or quantity < 1:
            raise ValueError('INVALID_QUANTITY')
        existing = next((x for x in self.lines if x.sku == sku), None)
        if existing:
            existing.quantity += quantity
        else:
            self.lines.append(CartLine(sku, quantity))

    def subtotal(self):
        return sum((self.catalog.get(x.sku).price * x.quantity for x in self.lines), D(0))


class Inventory:
    def __init__(self, catalog):
        self.catalog = catalog

    def reserve(self, lines):
        for line in lines:
            if self.catalog.get(line.sku).stock < line.quantity:
                raise ValueError('OUT_OF_STOCK')
        for line in lines:
            self.catalog.get(line.sku).stock -= line.quantity

    def release(self, lines):
        for line in lines:
            self.catalog.get(line.sku).stock += line.quantity


class Promotions:
    def __init__(self):
        self.codes = {'SAVE10': Voucher('SAVE10', 10, D(500000), D(100000))}

    def discount(self, code, subtotal, customer):
        if not code:
            return D(0)
        v = self.codes.get(code)
        if not v or not v.active:
            raise ValueError('INVALID_VOUCHER')
        if not customer.authenticated:
            raise ValueError('LOGIN_REQUIRED')
        if code in customer.used_codes:
            raise ValueError('VOUCHER_USED')
        # Validate minimum subtotal for voucher eligibility.
        if subtotal <= v.min_subtotal:
            raise ValueError('MIN_SUBTOTAL_NOT_MET')
        return min(money(subtotal * v.percent / D(100)), v.cap)


class Shipping:
    def calculate(self, subtotal, area):
        if area not in ('inner', 'outer'):
            raise ValueError('INVALID_AREA')
        return D(0) if area == 'inner' and subtotal >= D(500000) else (D(30000) if area == 'inner' else D(50000))


class Payments:
    def __init__(self):
        self.charges = []

    def collect(self, amount, method, approved, key):
        if method == 'COD':
            return 'PENDING_COD'
        if method != 'ONLINE':
            raise ValueError('INVALID_PAYMENT_METHOD')
        if not approved:
            raise ValueError('PAYMENT_DECLINED')
        self.charges.append({'key': key, 'amount': amount})
        return 'PAID'


class Checkout:
    def __init__(self, catalog=None):
        self.catalog = catalog or Catalog()
        self.inventory = Inventory(self.catalog)
        self.promotions = Promotions()
        self.shipping = Shipping()
        self.payments = Payments()
        self.orders = {}
        self.by_key = {}
        self.counter = 0

    def place(self, customer, cart, area='inner', voucher=None, method='COD', approved=True, key='REQ-001'):
        if not customer.authenticated:
            raise ValueError('LOGIN_REQUIRED')
        if not cart.lines:
            raise ValueError('EMPTY_CART')
        subtotal = cart.subtotal()
        discount = self.promotions.discount(voucher, subtotal, customer)
        shipping = self.shipping.calculate(subtotal, area)
        total = max(D(0), subtotal - discount + shipping)
        # Reserve stock during the checkout transaction.
        self.inventory.reserve(cart.lines)
        if key in self.by_key:
            return self.orders[self.by_key[key]]
        # Process the online payment outcome.
        payment_state = self.payments.collect(total, method, approved, key)
        self.counter += 1
        order = Order(f'ORD-{self.counter:04d}', customer.customer_id, total, payment_state,
                      [(x.sku, x.quantity) for x in cart.lines], discount, shipping, method)
        self.orders[order.order_id] = order
        self.by_key[key] = order.order_id
        if voucher:
            customer.used_codes.add(voucher)
        return order


def fixture(subtotal_case='600k'):
    service = Checkout()
    customer = Customer('U_NEW')
    cart = Cart(service.catalog)
    if subtotal_case == '500k':
        cart.add('A', 1); cart.add('B', 1)
    elif subtotal_case == '499999':
        cart.add('A', 1); cart.add('B', 1)
        service.catalog.products['B'].price = D(299999)
    elif subtotal_case == '1000k':
        cart.add('A', 5)
    elif subtotal_case == '1500k':
        cart.add('B', 5)
    elif subtotal_case == '600k':
        cart.add('A', 3)
    elif subtotal_case == '300k':
        cart.add('B', 1)
    else:
        raise ValueError('UNKNOWN_FIXTURE')
    return service, customer, cart


In [ ]:
%%writefile test_runner.py
"""STLab02 - CSV-driven test runner for EduCommerce (PASS / FAIL / BLOCKED)."""
import csv, json, time, datetime, html, base64, io
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from ecommerce_app import fixture, D, CartLine

REQUIRED = ['test_case_id', 'requirement_id', 'condition_id', 'scenario_id', 'title', 'priority',
            'precondition', 'fixture_key', 'input_json', 'expected_type', 'expected_json']


class Blocked(Exception):
    """The TEST cannot run (bad data / setup). Not a product defect."""


def load_cases(csv_path):
    """E04 - read CSV, reject duplicate IDs."""
    with Path(csv_path).open(newline='', encoding='utf-8-sig') as f:
        rows = list(csv.DictReader(f))
    ids = [r.get('test_case_id', '') for r in rows]
    dup = sorted({i for i in ids if ids.count(i) > 1})
    if dup:
        raise ValueError(f'DUPLICATE_TEST_IDS:{dup}')
    return rows


def build_env(fixture_key, setup):
    """E03 - fresh Checkout/Customer/Cart for EVERY test case (test isolation)."""
    try:
        service, customer, cart = fixture(fixture_key)
    except ValueError as exc:
        raise Blocked(f'FIXTURE_ERROR:{exc}')
    products = service.catalog.products
    for sku in setup.get('deactivate', []):
        products[sku].active = False
    for sku, qty in setup.get('stock', {}).items():
        products[sku].stock = qty
    if 'cart' in setup:
        cart.lines = [CartLine(s, q) for s, q in setup['cart']]
    if 'used_codes' in setup:
        customer.used_codes = set(setup['used_codes'])
    return service, customer, cart


def stocks(service):
    return {k: p.stock for k, p in service.catalog.products.items()}


def run_operation(inp, service, customer, cart):
    """E05 - dispatcher: call the target function, capture result AND state."""
    op, obs, before = inp['op'], {'error': ''}, stocks(service)
    try:
        if op == 'cart':
            for sku, qty in inp['adds']:
                cart.add(sku, qty)
        elif op == 'shipping':
            obs['shipping'] = int(service.shipping.calculate(D(inp['subtotal']), inp['area']))
        elif op == 'place':
            for call in inp['calls']:
                try:
                    service.place(customer, cart, area=call.get('area', 'inner'), voucher=call.get('voucher'),
                                  method=call.get('method', 'COD'), approved=call.get('approved', True),
                                  key=call.get('key', 'K-' + inp['_id']))
                    obs['error'] = ''
                except ValueError as exc:
                    obs['error'] = str(exc)
    except ValueError as exc:
        obs['error'] = str(exc)
    except Exception as exc:
        obs['error'] = f'UNEXPECTED_EXCEPTION:{type(exc).__name__}'
    obs['lines'] = [[l.sku, l.quantity] for l in cart.lines]
    obs['subtotal'] = int(cart.subtotal())
    obs['orders_count'] = len(service.orders)
    obs['charges_count'] = len(service.payments.charges)
    obs['used_codes'] = sorted(customer.used_codes)
    obs['stock_after'] = stocks(service)
    obs['stock_unchanged'] = before == obs['stock_after']
    return json.loads(json.dumps(obs)), before


def execute_case(row):
    """E06 - compare expected vs actual, handle exceptions, set PASS / FAIL / BLOCKED."""
    t0 = time.perf_counter()
    res = {k: row.get(k, '') for k in REQUIRED}
    res.update(actual_result='', error_message='', execution_status='', stock_before='', stock_after='')
    try:
        missing = [c for c in REQUIRED if not str(row.get(c, '')).strip()]
        if missing:
            raise Blocked(f'MISSING_COLUMNS:{missing}')
        try:
            inp, exp = json.loads(row['input_json']), json.loads(row['expected_json'])
        except json.JSONDecodeError as exc:
            raise Blocked(f'BAD_JSON:{exc.msg}')
        if inp.get('op') not in ('cart', 'shipping', 'place'):
            raise Blocked(f'UNKNOWN_OPERATION:{inp.get("op")}')
        inp['_id'] = row['test_case_id']
        service, customer, cart = build_env(row['fixture_key'], inp.get('setup', {}))
        obs, before = run_operation(inp, service, customer, cart)
        unknown = [k for k in exp if k not in obs]
        if unknown:
            raise Blocked(f'UNSUPPORTED_EXPECTED_KEY:{unknown}')
        diffs = {k: {'expected': v, 'actual': obs[k]} for k, v in exp.items() if obs[k] != v}
        res['actual_result'] = json.dumps({k: obs[k] for k in exp}, separators=(',', ':'))
        res['execution_status'] = 'PASS' if not diffs else 'FAIL'
        res['error_message'] = json.dumps(diffs, separators=(',', ':')) if diffs else ''
        res['stock_before'], res['stock_after'] = json.dumps(before), json.dumps(obs['stock_after'])
    except Blocked as exc:
        res['execution_status'], res['error_message'] = 'BLOCKED', str(exc)
    res['execution_time_ms'] = round((time.perf_counter() - t0) * 1000, 3)
    res['executed_at'] = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')
    return res


def run_suite(csv_path='test_cases.csv', out_dir='reports'):
    """Run all cases, export CSV + HTML dashboard (table + chart)."""
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    rows = load_cases(csv_path)
    run_id = 'RUN-' + datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
    results = []
    for i, r in enumerate(rows, 1):
        res = execute_case(r); res['run_id'] = f'{run_id}-{i:03d}'; results.append(res)
    df = pd.DataFrame(results)
    n = df.execution_status.value_counts()
    total, p, f_, b = len(df), int(n.get('PASS', 0)), int(n.get('FAIL', 0)), int(n.get('BLOCKED', 0))
    summ = df.groupby(['requirement_id', 'execution_status']).size().unstack(fill_value=0)
    for st in ('PASS', 'FAIL', 'BLOCKED'):
        if st not in summ: summ[st] = 0
    summ = summ[['PASS', 'FAIL', 'BLOCKED']]
    df.to_csv(out / 'test_execution.csv', index=False, encoding='utf-8-sig')
    fig, ax = plt.subplots(figsize=(7, 4)); bottom = None
    for st, col in (('PASS', '#16a34a'), ('FAIL', '#dc2626'), ('BLOCKED', '#f59e0b')):
        ax.bar(summ.index, summ[st], bottom=bottom, label=st, color=col)
        bottom = summ[st] if bottom is None else bottom + summ[st]
    ax.set_title('Results by requirement'); ax.set_ylabel('Test cases'); ax.legend(); fig.tight_layout()
    fig.savefig(out / 'chart_by_requirement.png', dpi=130)
    buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=100); plt.close(fig)
    b64 = base64.b64encode(buf.getvalue()).decode()
    colr = {'PASS': '#dcfce7', 'FAIL': '#fee2e2', 'BLOCKED': '#fef3c7'}
    body = ''.join(f'<tr style="background:{colr[r.execution_status]}">' + ''.join(
        f'<td>{html.escape(str(getattr(r, c)))}</td>' for c in
        ['test_case_id', 'requirement_id', 'scenario_id', 'expected_json', 'actual_result', 'execution_status']) + '</tr>'
        for r in df.itertuples())
    rq = ''.join(f'<tr><td>{i}</td><td>{x.PASS}</td><td>{x.FAIL}</td><td>{x.BLOCKED}</td></tr>' for i, x in summ.iterrows())
    page = f'''<!DOCTYPE html><html><head><meta charset="utf-8"><title>EduCommerce Test Report</title><style>
body{{font:14px Arial;margin:30px}}table{{border-collapse:collapse;width:100%;margin-bottom:20px}}td,th{{border:1px solid #cbd5e1;padding:6px;font-size:12px;text-align:left}}
th{{background:#16385e;color:#fff}}.m{{display:inline-block;margin:6px;padding:10px 16px;border-radius:10px;background:#edf3f9;font-size:18px}}</style></head><body>
<h1>EduCommerce - STLab02 Test Execution</h1><div>{run_id}</div>
<div class="m">Total: {total}</div><div class="m">PASS: {p}</div><div class="m">FAIL: {f_}</div><div class="m">BLOCKED: {b}</div>
<div class="m">Pass rate: {round(100*p/total,1)}%</div><br><img src="data:image/png;base64,{b64}">
<h2>Summary by requirement</h2><table><tr><th>REQ</th><th>PASS</th><th>FAIL</th><th>BLOCKED</th></tr>{rq}</table>
<h2>Execution</h2><table><tr><th>Case</th><th>REQ</th><th>Scenario</th><th>Expected</th><th>Actual</th><th>Status</th></tr>{body}</table></body></html>'''
    (out / 'test_dashboard.html').write_text(page, encoding='utf-8')
    print(f'{run_id} | CSV rows {len(rows)} | Executed {total} | PASS {p} | FAIL {f_} | BLOCKED {b} | pass rate {round(100*p/total,1)}%')
    bad = df[df.execution_status != 'PASS']
    if len(bad):
        print(bad[['test_case_id', 'requirement_id', 'execution_status', 'error_message']].to_string(index=False))
    return df, out


if __name__ == '__main__':
    run_suite()


In [ ]:
%%writefile test_cases.csv
test_case_id,requirement_id,condition_id,scenario_id,title,priority,precondition,fixture_key,input_json,expected_type,expected_json,execution_status,actual_result,error_message,evidence_reference
TC-ATM-001,REQ-15,COND-ATM-01,TS-ATM-01,"A=2 ok, B=100 insufficient: reject all, A unchanged",P0,"Stock A20,B15,C12; cart A2,B100",300k,"{""op"":""place"",""setup"":{""cart"":[[""A"",2],[""B"",100]]},""calls"":[{}]}",error+state,"{""error"":""OUT_OF_STOCK"",""orders_count"":0,""stock_after"":{""A"":20,""B"":15,""C"":12},""stock_unchanged"":true}",,,,
TC-ATM-002,REQ-15,COND-ATM-01,TS-ATM-02,"First SKU short (A=25 > 20): reject all, B unchanged",P1,"Cart A25,B1",300k,"{""op"":""place"",""setup"":{""cart"":[[""A"",25],[""B"",1]]},""calls"":[{}]}",error+state,"{""error"":""OUT_OF_STOCK"",""stock_after"":{""A"":20,""B"":15,""C"":12}}",,,,
TC-ATM-003,REQ-15,COND-ATM-02,TS-ATM-03,All SKUs enough: all stocks deducted,P0,"Cart A2,B3",300k,"{""op"":""place"",""setup"":{""cart"":[[""A"",2],[""B"",3]]},""calls"":[{}]}",value+state,"{""error"":"""",""orders_count"":1,""stock_after"":{""A"":18,""B"":12,""C"":12}}",,,,
TC-CRT-001,REQ-02,COND-CRT-01,TS-CRT-01,Add valid quantity 1 (lower boundary),P1,Empty cart; authenticated customer,300k,"{""op"":""cart"",""adds"":[[""A"",1]],""setup"":{""cart"":[]}}",state,"{""error"":"""",""lines"":[[""A"",1]]}",,,,
TC-CRT-002,REQ-02,COND-CRT-02,TS-CRT-02,Quantity 0 rejected (boundary),P0,Empty cart; authenticated customer,300k,"{""op"":""cart"",""adds"":[[""A"",0]],""setup"":{""cart"":[]}}",error+state,"{""error"":""INVALID_QUANTITY"",""lines"":[]}",,,,
TC-CRT-003,REQ-02,COND-CRT-02,TS-CRT-02,Negative quantity -1 rejected,P1,Empty cart; authenticated customer,300k,"{""op"":""cart"",""adds"":[[""A"",-1]],""setup"":{""cart"":[]}}",error+state,"{""error"":""INVALID_QUANTITY"",""lines"":[]}",,,,
TC-CRT-004,REQ-02,COND-CRT-03,TS-CRT-03,Same SKU added twice (2 then 3) is merged into one line qty 5,P0,Empty cart; authenticated customer,300k,"{""op"":""cart"",""adds"":[[""A"",2],[""A"",3]],""setup"":{""cart"":[]}}",state,"{""error"":"""",""lines"":[[""A"",5]],""subtotal"":1000000}",,,,
TC-SAF-001,REQ-16,COND-SAF-01,TS-SAF-01,method=WALLET rejected with INVALID_PAYMENT_METHOD,P0,Customer authenticated; fresh fixture,600k,"{""op"":""place"",""calls"":[{""method"":""WALLET""}]}",error+state,"{""error"":""INVALID_PAYMENT_METHOD"",""orders_count"":0,""charges_count"":0}",,,,
TC-SAF-002,REQ-16,COND-SAF-02,TS-SAF-02,WALLET rejected: stock must remain unchanged,P0,"Stock A20,B15,C12; cart A x3",600k,"{""op"":""place"",""calls"":[{""method"":""WALLET""}]}",state,"{""error"":""INVALID_PAYMENT_METHOD"",""stock_unchanged"":true}",,,,
TC-SAF-003,REQ-16,COND-SAF-02,TS-SAF-02,BANK_TRANSFER rejected with 2-line cart: all stock unchanged,P0,"Cart A1,B1 (500k)",500k,"{""op"":""place"",""calls"":[{""method"":""BANK_TRANSFER""}]}",state,"{""error"":""INVALID_PAYMENT_METHOD"",""stock_unchanged"":true,""stock_after"":{""A"":20,""B"":15,""C"":12}}",,,,
TC-SHP-001,REQ-09,COND-SHP-01,TS-SHP-01,"Inner 499,999 -> 30,000 (just below threshold)",P0,Shipping.calculate called directly,300k,"{""op"":""shipping"",""subtotal"":499999,""area"":""inner""}",boundary,"{""error"":"""",""shipping"":30000}",,,,
TC-SHP-002,REQ-09,COND-SHP-01,TS-SHP-01,"Inner 500,000 -> free (on threshold)",P0,Shipping.calculate called directly,300k,"{""op"":""shipping"",""subtotal"":500000,""area"":""inner""}",boundary,"{""error"":"""",""shipping"":0}",,,,
TC-SHP-003,REQ-09,COND-SHP-01,TS-SHP-01,"Inner 500,001 -> free (just above threshold)",P1,Shipping.calculate called directly,300k,"{""op"":""shipping"",""subtotal"":500001,""area"":""inner""}",boundary,"{""error"":"""",""shipping"":0}",,,,
TC-SHP-004,REQ-09,COND-SHP-02,TS-SHP-02,"Outer 600,000 -> 50,000 (no free ship outer)",P0,Shipping.calculate called directly,300k,"{""op"":""shipping"",""subtotal"":600000,""area"":""outer""}",value,"{""error"":"""",""shipping"":50000}",,,,
TC-VOU-001,REQ-07,COND-VOU-01,TS-VOU-01,"Unknown code FAKE99 rejected, no side effect",P0,Customer authenticated; fresh fixture,600k,"{""op"":""place"",""calls"":[{""voucher"":""FAKE99""}]}",error+state,"{""error"":""INVALID_VOUCHER"",""orders_count"":0,""stock_unchanged"":true}",,,,
TC-VOU-002,REQ-07,COND-VOU-02,TS-VOU-02,"Already-used SAVE10 rejected, no order, stock intact",P0,used_codes already contains SAVE10,600k,"{""op"":""place"",""setup"":{""used_codes"":[""SAVE10""]},""calls"":[{""voucher"":""SAVE10""}]}",error+state,"{""error"":""VOUCHER_USED"",""orders_count"":0,""stock_unchanged"":true}",,,,
TC-VOU-003,REQ-07,COND-VOU-03,TS-VOU-03,Voucher NOT marked used when online payment declined,P0,Customer authenticated; fresh fixture,600k,"{""op"":""place"",""calls"":[{""voucher"":""SAVE10"",""method"":""ONLINE"",""approved"":false}]}",error+state,"{""error"":""PAYMENT_DECLINED"",""used_codes"":[],""orders_count"":0}",,,,


In [ ]:
# [E01] Cấu trúc file
!ls -lh

In [ ]:
# [E07] Chạy toàn bộ test từ CSV
from test_runner import run_suite
results, out = run_suite('test_cases.csv', 'reports')
results[['test_case_id','requirement_id','scenario_id','expected_json','actual_result','execution_status','execution_time_ms']]

In [ ]:
# [E08] Dashboard: bảng + chart
from IPython.display import HTML, display
display(HTML(open('reports/test_dashboard.html', encoding='utf-8').read()))

In [ ]:
# [E03] fixture + setup
import inspect, test_runner
print(inspect.getsource(test_runner.build_env))

In [ ]:
# [E04] Đọc CSV + kiểm tra input
print(inspect.getsource(test_runner.load_cases))

In [ ]:
# [E05] Dispatcher gọi hàm nghiệp vụ
print(inspect.getsource(test_runner.run_operation))

In [ ]:
# [E06] So sánh expected/actual, try/except, PASS/FAIL/BLOCKED
print(inspect.getsource(test_runner.execute_case))

In [ ]:
# [E09] Các case FAIL + code nghiệp vụ nghi vấn
import ecommerce_app
fails = results[results.execution_status == 'FAIL']
print(fails[['test_case_id','expected_json','actual_result','stock_before','stock_after']].to_string(index=False))
print()
print(inspect.getsource(ecommerce_app.Checkout.place))

In [ ]:
# Tải thư mục reports về máy (CSV + dashboard HTML + chart)
!zip -qr reports.zip reports
from google.colab import files
files.download('reports.zip')